In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Explicit schema matching your exact CSV file columns
LOGISTICS_SCHEMA = StructType([
    StructField("Transaction_ID", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Origin_Facility", StringType(), True),
    StructField("Destination_City", StringType(), True),
    StructField("Vehicle_Type", StringType(), True),
    StructField("Route_Type", StringType(), True),
    StructField("Distance_KM", DoubleType(), True),
    StructField("Package_Weight_KG", DoubleType(), True),
    StructField("Traffic_Conditions", StringType(), True),
    StructField("Carbon_Emission_kgCO2e", DoubleType(), True),
    StructField("Is_Eco_Friendly", IntegerType(), True)
])

# Secure Unity Catalog paths!
RAW_DATA_PATH = "/Volumes/workspace/default/logistics_data/ecommerce_logistics_carbon_emissions_v1.csv" 
GOLD_DATA_PATH = "/Volumes/workspace/default/logistics_data/gold_emissions/"

print("Configurations and Schema loaded successfully.")

Configurations and Schema loaded successfully.


In [0]:
from pyspark.sql.functions import col, round, when

def clean_data(df):
    """Removes duplicate transactions and drops rows with missing distance or weight."""
    return df.dropDuplicates(["Transaction_ID"]) \
             .dropna(subset=["Distance_KM", "Package_Weight_KG"]) \
             .filter(col("Distance_KM") > 0)

def engineer_features(df):
    """Calculates advanced efficiency metrics for downstream ML models."""
    df_final = df.withColumn(
        "Emission_per_KM",
        round(col("Carbon_Emission_kgCO2e") / col("Distance_KM"), 4)
    ).withColumn(
        "Emission_per_KG",
        when(col("Package_Weight_KG") > 0, round(col("Carbon_Emission_kgCO2e") / col("Package_Weight_KG"), 4))
        .otherwise(0.0)
    )
    return df_final

print("Transformation functions registered.")

Transformation functions registered.


In [0]:
# 1. Ingestion
print("Reading raw data from Unity Catalog Volume...")
raw_df = spark.read.csv(RAW_DATA_PATH, header=True, schema=LOGISTICS_SCHEMA)

# 2. Transformation
print("Applying data cleansing and feature engineering...")
clean_df = clean_data(raw_df)
final_df = engineer_features(clean_df)

# 3. Quick Visual Check (Databricks' 'display' renders a beautiful HTML table)
print("Previewing transformed data:")
display(final_df.limit(5))

# 4. Loading to Gold Layer (Partitioned Parquet)
print("Writing optimized Parquet files to Unity Catalog...")
final_df.write.mode("overwrite") \
        .partitionBy("Vehicle_Type") \
        .parquet(GOLD_DATA_PATH)

print("ETL Pipeline Execution Complete! Gold data is safely partitioned and stored.")

Reading raw data from Unity Catalog Volume...
Applying data cleansing and feature engineering...
Previewing transformed data:


Transaction_ID,Date,Origin_Facility,Destination_City,Vehicle_Type,Route_Type,Distance_KM,Package_Weight_KG,Traffic_Conditions,Carbon_Emission_kgCO2e,Is_Eco_Friendly,Emission_per_KM,Emission_per_KG
TRX-0E8FDACD,2025-04-14,Makassar Port,Owensfort,Heavy Truck,Inter-City,772.5,4116.0,High,911.44,0,1.1799,0.2214
TRX-E9442C68,2025-03-04,Jakarta Fulfillment Center,West Elizabethshire,Diesel Van (Euro 4),Mixed Route,10.0,17.6,High,3.587,0,0.3587,0.2038
TRX-8C7844AD,2025-04-02,Surabaya Warehouse,North Nataliemouth,Motorcycle (Courier),Inter-District,23.7,26.1,High,3.235,0,0.1365,0.1239
TRX-52A6900C,2025-03-04,Semarang Depot,Annashire,Motorcycle (Courier),Inter-District,40.0,24.5,Normal,4.167,0,0.1042,0.1701
TRX-5B22FF5F,2025-10-24,Semarang Depot,Jenniferburgh,Heavy Truck,Inter-City,648.3,4322.7,Normal,625.195,0,0.9644,0.1446


Writing optimized Parquet files to Unity Catalog...
ETL Pipeline Execution Complete! Gold data is safely partitioned and stored.


In [0]:
# Prove the data was written and that partition pruning works
print("Validating Gold Layer Data via SQL...")

validation_df = spark.read.parquet(GOLD_DATA_PATH)
validation_df.createOrReplaceTempView("gold_emissions")

# Run an aggregation query to show off the finalized data
display(spark.sql("""
    SELECT 
        Vehicle_Type, 
        COUNT(*) as Total_Deliveries,
        ROUND(AVG(Carbon_Emission_kgCO2e), 2) as Avg_CO2_Emissions,
        ROUND(AVG(Emission_per_KM), 4) as Avg_Emission_per_KM
    FROM gold_emissions
    GROUP BY Vehicle_Type
    ORDER BY Avg_CO2_Emissions DESC
"""))

Validating Gold Layer Data via SQL...


Vehicle_Type,Total_Deliveries,Avg_CO2_Emissions,Avg_Emission_per_KM
Heavy Truck,1679,463.14,1.0354
Diesel Van (Euro 4),1743,25.91,0.3168
Diesel Van (Euro 6),1731,17.12,0.2148
Electric Van (EV),1750,4.06,0.0515
Motorcycle (Courier),1723,2.51,0.112
Drone Delivery,1664,0.06,0.0082
Cargo Bicycle,1710,0.0,0.0
